In [ ]:
!kaggle datasets download konradb/atticus-open-contract-dataset-aok-beta

import zipfile
import os

zip_path = "atticus-open-contract-dataset-aok-beta.zip"
extract_dir = "atticus_open_contract_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Extracted '{zip_path}' to directory '{extract_dir}'")

Dataset URL: https://www.kaggle.com/datasets/konradb/atticus-open-contract-dataset-aok-beta
License(s): Attribution 4.0 International (CC BY 4.0)
  0%|                                                | 0.00/102M [00:00<?, ?B/s]
100%|████████████████████████████████████████| 102M/102M [00:00<00:00, 1.38GB/s]


In [10]:
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy
import json
import os
import re

# Load spaCy English language model
nlp = spacy.load("en_core_web_sm")

# I/O directories for assignment
os.makedirs("input", exist_ok=True)
os.makedirs("output", exist_ok=True)

In [11]:
def clean_legal_text(text):
    """Removes redactions, page numbers, and formatting artifacts."""
    text = re.sub(r'\[\*\*\*\]', '', text)
    text = re.sub(r'Page \d+', '', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def extract_clauses(text):
    """
    Splits text into semantically independent clauses.
    """
    clean_text = clean_legal_text(text)
    doc = nlp(clean_text)
    
    clauses = []
    start_idx = 0
    
    for token in doc:
        split_here = False

        # Standard sentence boundaries and legal list punctuation
        if token.text in ['.', ';']:
            split_here = True
            
        # Only split on colon if followed by open paren
        elif token.text == ':':
            if token.i + 1 < len(doc) and doc[token.i + 1].text == '(':
                split_here = True
                
        # Conjunctions: only split if new subject present in second verb
        elif token.dep_ == 'cc':
            head = token.head
            if head.pos_ in ['VERB', 'AUX']:
                for child in head.children:
                    if child.dep_ == 'conj' and child.pos_ in ['VERB', 'AUX']:
                        has_subject = any(c.dep_ in ['nsubj', 'nsubjpass'] for c in child.children)
                        if has_subject:
                            split_here = True
                            break

        if split_here:
            # Ensure punctuation remains attached to the clause
            end_idx = token.i + 1 if token.text in ['.', ';', ':'] else token.i

            clause_span = doc[start_idx:end_idx].text.strip()
            
            if len(clause_span) > 10:
                # Remove any trailing comma
                clause_span = re.sub(r'[,]\s*$', '', clause_span)
                
                # Ensure the clause ends with proper punctuation
                if not clause_span.endswith(('.', ';', ':')):
                    clause_span += "."
                    
                clauses.append(clause_span)
            
            # Move the start pointer for the next clause
            start_idx = end_idx

    # Capture trailing text at end if any
    if start_idx < len(doc):
        final_span = doc[start_idx:].text.strip()
        if len(final_span) > 10:
            if not final_span.endswith('.'):
                final_span += "."
            clauses.append(final_span)
            
    return clauses

def process_task_1_1(input_filepath, output_filepath):
    with open(input_filepath, 'r', encoding='utf-8') as f:
        raw_text = f.read()
        
    clauses = extract_clauses(raw_text)
    
    with open(output_filepath, 'w', encoding='utf-8') as f:
        for clause in clauses:
            f.write(clause + '\n')
            
    print(f"[Task 1.1] Extracted {len(clauses)} clauses. Saved to {output_filepath}")
    return clauses

In [ ]:
def generate_iob_tags(clause):
    """
    Detects noun phrases and applies strict IOB tagging.
    Fixes the "broken chunk" problem by ensuring B-NP is applied
    after any token that gets forced to 'O'.
    """
    doc = nlp(clause)
    iob_tags = ["O"] * len(doc)
    
    for chunk in doc.noun_chunks:
        is_beginning = True
        for token in chunk:
            # Prevent tokens not suitable as NP (e.g., punctuation, preps, etc.) from being chunked
            is_excluded = (
                token.is_punct or 
                token.is_space or 
                token.tag_ in ["WDT", "WP"] or
                token.pos_ in ["ADP", "SCONJ"]
            )
            if is_excluded:
                iob_tags[token.i] = "O"
                # The phrase is broken, so the next valid word MUST be a B-NP
                is_beginning = True 
            else:
                if is_beginning:
                    iob_tags[token.i] = "B-NP"
                    is_beginning = False
                else:
                    iob_tags[token.i] = "I-NP"
    
    formatted_output = []
    for token in doc:
        clean_token = token.text.strip()
        if clean_token:
            formatted_output.append(f"{clean_token}\t{iob_tags[token.i]}")
    return formatted_output

def process_task_1_2(clauses, output_filepath="output/chunks.txt"):
    """
    Iterates through extracted clauses and writes the IOB-tagged tokens
    to a file, ensuring a blank line separator between clauses.
    """
    with open(output_filepath, 'w', encoding='utf-8') as f:
        for clause in clauses:
            iob_tags = generate_iob_tags(clause)
            if iob_tags:
                f.write("\n".join(iob_tags) + "\n\n")  # Blank line to separate clauses
            
    print(f"[Task 1.2] IOB Noun Phrase chunking completed. Saved to {output_filepath}")

In [13]:
def extract_dependencies(clause):
    """Extracts syntactic dependencies for each token."""
    doc = nlp(clause)
    clause_deps = []
    
    for token in doc:
        if token.text.strip():
            dep_info = {
                "Token": token.text,
                "Head": token.head.text,
                "Dependency": token.dep_
            }
            clause_deps.append(dep_info)
            
    return clause_deps

def process_task_1_3(clauses, output_filepath):
    all_dependencies = []
    
    for clause in clauses:
        all_dependencies.append({
            "Clause": clause,
            "Dependencies": extract_dependencies(clause)
        })
        
    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(all_dependencies, f, indent=4)
        
    print(f"[Task 1.3] Dependency parsing completed. Saved to {output_filepath}")

In [16]:
datasets = [
    "SPONSORSHIP_AGREEMENT.txt",
    "DISTRIBUTOR_AGREEMENT.txt",
    "SOFTWARE_LICENSE_AGREEMENT.txt"
]

for filename in datasets:
    input_path = os.path.join("input", filename)
    
    base_name = filename.replace(".txt", "")
    clauses_out = os.path.join("output", f"{base_name}_clauses.txt")
    chunks_out = os.path.join("output", f"{base_name}_chunks.txt")
    deps_out = os.path.join("output", f"{base_name}_dependency.json")
    
    # Skip if input file is missing
    if not os.path.exists(input_path):
        print(f"Warning: Please place {filename} in the 'input' directory.")
        continue
        
    print(f"--- Processing: {filename} ---")
    
    clauses = process_task_1_1(input_path, clauses_out)
    process_task_1_2(clauses, chunks_out)
    process_task_1_3(clauses, deps_out)
    
    print("\n")

--- Processing: SPONSORSHIP_AGREEMENT.txt ---
[Task 1.1] Extracted 539 clauses. Saved to output/SPONSORSHIP_AGREEMENT_clauses.txt
[Task 1.2] IOB Noun Phrase chunking completed. Saved to output/SPONSORSHIP_AGREEMENT_chunks.txt
[Task 1.3] Dependency parsing completed. Saved to output/SPONSORSHIP_AGREEMENT_dependency.json


--- Processing: DISTRIBUTOR_AGREEMENT.txt ---
[Task 1.1] Extracted 320 clauses. Saved to output/DISTRIBUTOR_AGREEMENT_clauses.txt
[Task 1.2] IOB Noun Phrase chunking completed. Saved to output/DISTRIBUTOR_AGREEMENT_chunks.txt
[Task 1.3] Dependency parsing completed. Saved to output/DISTRIBUTOR_AGREEMENT_dependency.json


--- Processing: SOFTWARE_LICENSE_AGREEMENT.txt ---
[Task 1.1] Extracted 241 clauses. Saved to output/SOFTWARE_LICENSE_AGREEMENT_clauses.txt
[Task 1.2] IOB Noun Phrase chunking completed. Saved to output/SOFTWARE_LICENSE_AGREEMENT_chunks.txt
[Task 1.3] Dependency parsing completed. Saved to output/SOFTWARE_LICENSE_AGREEMENT_dependency.json


